# Executive Function Task Performance, System Usability, and Thematic Annotations in Undergraduate Students Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the *Executive Function Task Performance, System Usability, and Thematic Annotations in Undergraduate Students* FAIRˆ² dataset using the `mlcroissant` library.

### Dataset Source
This dataset is described by the Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.f2na-3dct/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant matplotlib seaborn --quiet

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset's Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.f2na-3dct/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")

## 2. Data Overview
The dataset holds one record set per cognitive task (one row per participant, with their demographics and sleep self-report), the End-User Computing Satisfaction (EUCS) survey of the platform, the thematic analysis of the participants' open-ended comments, and ten derived, analysis-ready summary tables (named by UUID).

In [ ]:
import unicodedata
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

record_sets = {rs.name: rs for rs in metadata.record_sets}
display(pd.DataFrame([{'Record set': name, 'Fields': len(rs.fields), 'Description': rs.description}
                      for name, rs in record_sets.items()])
        .style.hide(axis='index').set_caption('Record sets'))

## 3. Data Extraction
Load the record sets of the six tasks, the EUCS survey and the thematic analysis as pandas DataFrames. `mlcroissant` names the columns by field `@id` and returns text as bytes: the columns get the field names and the text is decoded. The task tables are then combined into one table, with one row per participant and task, and the Spanish labels of the source are translated.

Each task classifies the participants as *Normal*, *Mild* or *Moderate* (impairment) from ranges of one of its scores: the errors for the Wisconsin card sorting test, the correct responses for the others (correct moves for the Tower of Hanoi).

The grade average and weekly study hours are free-text answers in the source: the grades are read as numbers, with "punto" as a decimal point and the grades above 10 (e.g. `83`) read as written without their decimal point (8.3).

In [ ]:
def decode(value):
    return value.decode('utf-8') if isinstance(value, bytes) else value

def load(name):
    record_set = record_sets[name]
    df = pd.DataFrame(dataset.records(record_set=record_set.id))
    return df.rename(columns={field.id: field.name for field in record_set.fields}).apply(lambda column: column.map(decode))

# Record set -> task, correct and error columns, and the score its categories are based on
TASKS = {
    'bloom_2_0_filtrados_n64/wisconsin_card_sorting_test_errors.csv': ('Wisconsin', 'correctas', 'errores', 'Errors'),
    'bloom_2_0_filtrados_n64/ravens_progressive_matrices_correct_responses.csv': ('Raven', 'correctas', 'errores', 'Correct responses'),
    'bloom_2_0_filtrados_n64/stroop_correct_responses.csv': ('Stroop', 'correctas', 'errores', 'Correct responses'),
    'bloom_2_0_filtrados_n64/token_test_correct_responses.csv': ('Token', 'correctas', 'errores', 'Correct responses'),
    'bloom_2_0_filtrados_n64/tower_of_hanoi_3_discs.csv': ('Tower of Hanoi (3)', 'movAcertados', 'movFallidos', 'Correct moves'),
    'bloom_2_0_filtrados_n64/tower_of_hanoi_4_discs.csv': ('Tower of Hanoi (4)', 'movAcertados', 'movFallidos', 'Correct moves'),
}
CATEGORIES = {'Normal': 'Normal', 'Leve': 'Mild', 'Moderada': 'Moderate'}
FREQUENCY = {'nunca': 'Never', 'rara vez': 'Rarely', 'a veces': 'Sometimes', 'frecuentemente': 'Frequently', 'siempre': 'Always'}
SLEEP = {
    'meCuestaConciliarElSueñoPorLasNoches': 'Difficulty falling asleep',
    'meDespiertoVariasVecesDuranteLaNoche': 'Waking up several times a night',
    'alDespertar,NoMeSientoDescansado/a': 'Not feeling rested on waking',
    'tengoDificultadesParaMantenerLaConcentraciónDuranteElDíaDebidoAlSueño': 'Daytime concentration difficulty',
}

task_tables, rows = {}, []
for name, (task, correct, errors, scored_on) in TASKS.items():
    df = task_tables[task] = load(name)
    rows.append(pd.DataFrame({
        'participant': df['identificacion'], 'task': task, 'scored_on': scored_on,
        'duration_s': df['duracion total (ms)'] / 1000, 'correct': df[correct], 'errors': df[errors],
        'category': df['categoría'].map(CATEGORIES), 'normal_range': df['rango normal'],
    }))
tasks = pd.concat(rows, ignore_index=True)

# The grade average and study hours are free text in the source: '8', '83' (for 8.3), '8 punto 5'
def grade(value):
    number = pd.to_numeric(str(value).strip().replace(' punto ', '.').replace(',', '.'), errors='coerce')
    return number / 10 if number > 10 else number   # a grade above 10 was written without its decimal point

# The participants: demographics and sleep self-report, the same in every task table
source = task_tables['Wisconsin']
participants = pd.DataFrame({
    'participant': source['identificacion'],
    'sex': source['sexo'].map({'masculino': 'Male', 'femenino': 'Female'}),
    # The faculty is written in two Unicode forms in the source: normalize it
    'faculty': source['escuelaOFacultad'].map(lambda s: unicodedata.normalize('NFC', s))
               .map({'facultad de ingeniería': 'Engineering', 'facultad de empresariales': 'Business'}),
    'grade_average': source['promedioActual(escala0-10)'].map(grade),
    'weekly_study_hours': pd.to_numeric(source['horasPromedioDeEstudioSemanal(escala0-60)'], errors='coerce'),
    **{label: source[column].map(FREQUENCY) for column, label in SLEEP.items()},
})
tasks = tasks.merge(participants[['participant', 'sex']], on='participant')

unreadable = participants[['grade_average', 'weekly_study_hours']].isna().sum()
if unreadable.any():
    print(f'Unreadable answers, left empty: {unreadable.to_dict()}')

eucs = load('bloom_2_0_eucs_n55/eucs_form_responses_1.csv')
thematic = load('bloom_2_0_thematic_analysis_n55.csv')

display(pd.DataFrame({
    'Rows': {'Task results (participants × tasks)': len(tasks), 'Participants': len(participants),
             'EUCS survey responses': len(eucs), 'Thematic analysis of the comments': len(thematic)},
}).style.format('{:,}').set_caption('Loaded data'))

## 4. Exploratory Data Analysis (EDA)
### 4.1 Participants

In [ ]:
profile = {
    'Participants': len(participants),
    **{f'{sex}': count for sex, count in participants['sex'].value_counts().items()},
    **{f'Faculty of {faculty}': count for faculty, count in participants['faculty'].value_counts().items()},
    'Grade average (0-10), mean': participants['grade_average'].mean(),
    'Weekly study hours, mean': participants['weekly_study_hours'].mean(),
}
profile = {label: f'{value:.1f}' if isinstance(value, float) else f'{value:,}' for label, value in profile.items()}
display(pd.DataFrame({'Value': profile}).style.set_properties(**{'text-align': 'right'}).set_caption('Participants'))

frequency = list(FREQUENCY.values())
sleep = pd.DataFrame({label: participants[label].value_counts() for label in SLEEP.values()}).T.reindex(columns=frequency).fillna(0).astype(int)
display(sleep.style.background_gradient(cmap='Blues', axis=None)
        .set_caption('Sleep self-report: number of participants by answer'))

### 4.2 Task performance
The scores of each task, and the share of participants in each performance category. The categories come from different scores (*Scored on*), with the normal range of that score; the shares are of the classified participants (a few have no category in the source).

In [ ]:
TASK_ORDER = [task for task, *_ in TASKS.values()]
summary = tasks.groupby('task').agg(**{
    'Scored on': ('scored_on', 'first'),
    'Normal range': ('normal_range', 'first'),
    'Median duration (s)': ('duration_s', 'median'),
    'Median correct': ('correct', 'median'),
    'Median errors': ('errors', 'median'),
}).reindex(TASK_ORDER)
shares = 100 * pd.crosstab(tasks['task'], tasks['category'], normalize='index').reindex(index=TASK_ORDER, columns=list(CATEGORIES.values())).fillna(0)
summary = summary.join(shares.rename(columns=lambda c: f'{c} (%)'))
summary['Not classified'] = tasks['category'].isna().groupby(tasks['task']).sum().reindex(TASK_ORDER)
summary.index.name = 'Task'

share_columns = [f'{c} (%)' for c in CATEGORIES.values()]
display(summary.style
        .format('{:.0f}', subset=['Median duration (s)', 'Median correct', 'Median errors'] + share_columns)
        .background_gradient(cmap='Blues', subset=share_columns, vmin=0, vmax=100)
        .set_caption('Task performance'))

All participants are in the normal range of the Stroop test, while nearly all of them make enough errors in the Wisconsin card sorting test to be classified with moderate impairment: the categories reflect each task's thresholds more than a comparable level of difficulty.

### 4.3 Grouping: performance by sex
The share of participants in the normal range of each task, by sex.

In [ ]:
normal_by_sex = (tasks.assign(normal=tasks['category'] == 'Normal')
                 .pivot_table(index='task', columns='sex', values='normal', aggfunc='mean')
                 .reindex(TASK_ORDER) * 100)
normal_by_sex.index.name = 'Task'
counts = participants['sex'].value_counts()
normal_by_sex.columns = [f'{sex} (n={counts[sex]}), % normal' for sex in normal_by_sex.columns]
display(normal_by_sex.style.format('{:.0f}').background_gradient(cmap='Blues', axis=None, vmin=0, vmax=100)
        .set_caption('Participants in the normal range, by sex (%)'))

### 4.4 Normalizing: an error score across tasks
The tasks count errors on very different scales (wrong moves of the Tower of Hanoi go up to 82, Stroop errors up to 14). To relate errors to sleep, each task's errors are standardized as a z-score within the task, and averaged over the six tasks for each participant: an **error score** of 0 is average, positive means more errors than the other participants.

In [ ]:
tasks['error_z'] = tasks.groupby('task')['errors'].transform(lambda s: (s - s.mean()) / s.std())
participants = participants.merge(tasks.groupby('participant')['error_z'].mean().rename('error_score'), on='participant')

CONCENTRATION = 'Daytime concentration difficulty'
by_concentration = (participants.groupby(CONCENTRATION)['error_score']
                    .agg(['size', 'mean', 'median']).reindex(frequency))
by_concentration.columns = ['Participants', 'Mean error score', 'Median error score']
display(by_concentration.style.format({'Participants': '{:.0f}', 'Mean error score': '{:+.2f}', 'Median error score': '{:+.2f}'})
        .set_caption('Error score by daytime concentration difficulty due to sleep'))

levels = participants[CONCENTRATION].map({label: rank for rank, label in enumerate(frequency)})
rho = levels.corr(participants['error_score'], method='spearman')
print(f'Spearman correlation between concentration difficulty and the error score: {rho:+.2f} (n={len(participants)})')

## 5. Visualization
- **Errors vs correct responses** of each participant in each task (for the Tower of Hanoi: wrong vs correct moves).
- **Performance categories** by task.
- **Task duration** by task, on a logarithmic scale: the tasks take from seconds to tens of minutes.
- **Sleep and errors**: the error score of the participants by how often they have difficulty concentrating during the day because of sleep.

In [ ]:
TASK_COLORS = dict(zip(TASK_ORDER, ['#2a78d6', '#eb6834', '#1baf7a', '#eda100', '#e87ba4', '#6250d6']))
CATEGORY_COLORS = {'Normal': '#cde2fb', 'Mild': '#6da7ec', 'Moderate': '#1c5cab'}   # light to dark: increasing impairment

fig, axes = plt.subplots(2, 2, figsize=(16, 11))

# Errors vs correct responses, by task
ax = axes[0, 0]
sns.scatterplot(data=tasks, x='correct', y='errors', hue='task', hue_order=TASK_ORDER, palette=TASK_COLORS,
                s=40, alpha=0.8, edgecolor='white', linewidth=0.5, ax=ax)
ax.set(title='Task performance: errors vs correct responses', xlabel='Correct responses (moves)', ylabel='Errors (wrong moves)')
ax.legend(title='Task', fontsize=9)

# Performance categories, by task (100% stacked)
ax = axes[0, 1]
category_counts = pd.crosstab(tasks['task'], tasks['category']).reindex(index=TASK_ORDER[::-1], columns=list(CATEGORY_COLORS)).fillna(0)
left = np.zeros(len(category_counts))
for category, color in CATEGORY_COLORS.items():
    values = category_counts[category].to_numpy()
    bars = ax.barh(category_counts.index, values, left=left, color=color, edgecolor='white', linewidth=2, label=category)
    ax.bar_label(bars, labels=[f'{v:.0f}' if v >= 4 else '' for v in values], label_type='center', fontsize=9,
                 color='#0b0b0b' if category == 'Normal' else 'white')
    left += values
ax.set(xlabel='Participants', xlim=(0, len(participants)))
ax.set_title('Performance category distribution', loc='left')
ax.legend(ncols=3, loc='lower right', bbox_to_anchor=(1, 1), frameon=False, fontsize=9)

# Task duration, by task
ax = axes[1, 0]
sns.boxplot(data=tasks, x='task', y='duration_s', hue='task', order=TASK_ORDER, palette=TASK_COLORS,
            legend=False, fliersize=3, ax=ax)
ax.set(title='Task duration distribution', xlabel='', ylabel='Duration (s, log scale)', yscale='log')
ax.tick_params(axis='x', rotation=20)

# Sleep and errors: error score by daytime concentration difficulty
ax = axes[1, 1]
sns.boxplot(data=participants, x=CONCENTRATION, y='error_score', order=frequency, color='#cde2fb',
            showfliers=False, ax=ax)
sns.stripplot(data=participants, x=CONCENTRATION, y='error_score', order=frequency, color='#1c5cab',
              size=5, jitter=0.15, ax=ax)
ax.axhline(0, color='#898781', linewidth=1, linestyle='--')
ax.set_xticks(range(len(frequency)), [f'{level}\n(n={by_concentration.loc[level, "Participants"]:.0f})' for level in frequency])
ax.set(title='Sleep quality and errors', xlabel='Difficulty concentrating during the day because of sleep',
       ylabel='Error score (mean z-score over the tasks)')

plt.tight_layout()
plt.show()

### 5.1 Usability of the platform
The participants rated the platform with the End-User Computing Satisfaction (EUCS) questionnaire, from 1 (lowest) to 5, and commented on it in two open-ended questions, annotated with themes. The fifteen questions are grouped here by the aspect they ask about.

In [ ]:
def dimension(question):
    q = question.lower()
    if 'a tiempo' in q or 'actualizada' in q:
        return 'Timeliness'
    if any(word in q for word in ('formato', 'clara', 'presentación', 'fácil de entender')):
        return 'Format'
    if any(word in q for word in ('preciso', 'precisión', 'fiable')):
        return 'Accuracy'
    return 'Content'

questions = [column for column in eucs.columns if column.startswith('¿') and pd.api.types.is_numeric_dtype(eucs[column])]
ratings = eucs[questions].melt(var_name='question', value_name='rating').dropna()
ratings['dimension'] = ratings['question'].map(dimension)
DIMENSIONS = ['Content', 'Accuracy', 'Format', 'Timeliness']

usability = ratings.groupby('dimension').agg(**{
    'Questions': ('question', 'nunique'), 'Mean rating': ('rating', 'mean'),
    'Rated 4 or 5 (%)': ('rating', lambda r: 100 * (r >= 4).mean())}).reindex(DIMENSIONS)
usability.index.name = 'EUCS dimension'
display(usability.style.format({'Mean rating': '{:.2f}', 'Rated 4 or 5 (%)': '{:.0f}'})
        .set_caption(f'Satisfaction with the platform ({len(eucs)} respondents)'))

themes = (thematic['themes'].dropna().str.split(';').explode().str.strip()
          .loc[lambda s: s != ''].value_counts())

RATING_COLORS = ['#cde2fb', '#9ec5f4', '#6da7ec', '#3987e5', '#1c5cab']
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
distribution = 100 * pd.crosstab(ratings['dimension'], ratings['rating'], normalize='index').reindex(index=DIMENSIONS[::-1], columns=range(1, 6)).fillna(0)
left = np.zeros(len(distribution))
for rating, color in zip(range(1, 6), RATING_COLORS):
    values = distribution[rating].to_numpy()
    bars = ax.barh(distribution.index, values, left=left, color=color, edgecolor='white', linewidth=2, label=str(rating))
    ax.bar_label(bars, labels=[f'{v:.0f}%' if v >= 6 else '' for v in values], label_type='center', fontsize=9,
                 color='#0b0b0b' if rating <= 2 else 'white')
    left += values
ax.set(title='EUCS ratings by dimension', xlabel='Share of the ratings (%)', xlim=(0, 100))
ax.legend(title='Rating', ncols=5, loc='lower center', bbox_to_anchor=(0.5, -0.32))

ax = axes[1]
bars = ax.barh(themes.index[::-1], themes.values[::-1], color='#2a78d6', edgecolor='white', linewidth=2)
ax.bar_label(bars, padding=3, fontsize=9)
ax.set(title="Themes of the participants' comments", xlabel='Respondents mentioning the theme')

plt.tight_layout()
plt.show()

## 6. Conclusion
We loaded the *Bloom 2.0* dataset with the `mlcroissant` library and explored the performance of 64 undergraduate students in six executive function tasks, their sleep self-report, and their satisfaction with the assessment platform:

- **Task performance**: the tasks differ widely in duration (from seconds for the Tower of Hanoi with 3 discs to over ten minutes for Raven's matrices) and in how participants are classified: everyone is in the normal range of the Stroop test, nearly everyone has moderate impairment in the Wisconsin card sorting test. The categories reflect each task's thresholds and are not comparable across tasks.
- **Sleep**: once errors are standardized within each task, the participants' error score shows no clear relation to their daytime concentration difficulty because of sleep (Spearman correlation close to zero). Only the three participants who always have that difficulty make clearly more errors, too few to draw conclusions.
- **Usability**: the platform is rated favourably in every EUCS dimension, and the comments mostly ask for clearer instructions.

Further exploration can use the ten derived summary tables of the dataset, relate the performance to the grade average and study hours, or compare the tasks within participants.